# 01 · Spatial operations: what they do, how they are tested, one real example (A24)

Three deterministic operations, written in `src/space_mel/operations.py` and tested in `tests/test_operations.py`
(run `.venv\Scripts\python.exe -m pytest` from the `spatial-reasoning` folder):

| Operation | Question it answers | Output |
|---|---|---|
| `spots_within_radius` | which spots are within *r* µm of this spot? | a list of spots |
| `distance_to_region` | how far is each spot from the nearest spot of a region? | one distance (µm) per spot |
| `neighbourhood_fraction` | around each spot, what share of its neighbours belong to a category? | a fraction per spot |

**Rules:** distances in µm; the boundary is inclusive (exactly *r* µm counts); a spot is not its own neighbour;
no neighbours → fraction NaN; no region in a section → distance NaN; spots of different tissue sections never
connect.

**Sections:** 1. the operations on a tiny drawn example · 2. why real coordinates need care · 3. a real example on
A24 · 4. summary.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "02_spatial_operations"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.operations import distance_to_region, neighbourhood_fraction, spots_table, spots_within_radius
from space_mel.spatial import grid_to_um

FIGS = STAGE / "results"
FIGS.mkdir(parents=True, exist_ok=True)

# colours: blue / orange for the two groups, grey for everything else, a light-to-dark blue for magnitudes
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#d9d9d6", "#52514e"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#2a78d6", "#1c5cab", "#0d366b"])
plt.rcParams.update({"figure.dpi": 90, "axes.titlesize": 11, "axes.titlelocation": "left"})


def spot_map(ax, spots, color, size=12, title="", **kw):
    """Draw spots at their µm positions (y downwards, like the tissue image)."""
    sc = ax.scatter(spots.x_um, spots.y_um, c=color, s=size, linewidths=0, **kw)
    ax.set_aspect("equal")
    if not ax.yaxis_inverted():
        ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)
    return sc

## 1. The three operations on a tiny drawn example

A small piece of the real Visium hexagonal grid (spot centres 100 µm apart) plus one isolated spot far away.
Every answer here can be checked by eye; the same cases are in the automatic tests.

In [ ]:
rows, cols = np.meshgrid(np.arange(-2, 3), np.arange(-4, 5), indexing="ij")
keep = (rows + cols) % 2 == 0                         # Visium uses every other (row, col) position
xy = grid_to_um(rows[keep], cols[keep])
toy = pd.DataFrame({"x_um": xy[:, 0], "y_um": xy[:, 1], "section": "toy"},
                   index=[f"r{r}c{c}" for r, c in zip(rows[keep], cols[keep])])
toy.loc["alone"] = [700.0, 0.0, "toy"]                 # isolated spot: no neighbours within 100 µm

centre = "r0c0"
region = pd.Series(toy.index.isin(["r-2c-4", "r-2c-2", "r-1c-3"]), index=toy.index)       # a small region
category = pd.Series(toy.index.isin(["r0c2", "r1c1", "r-1c1", "r1c-1"]), index=toy.index)  # a category

nb = spots_within_radius(toy, centre, 100)
dist = distance_to_region(toy, region)
frac = neighbourhood_fraction(toy, category, 100)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"wspace": 0.15})
xy_centre = toy.loc[centre, ["x_um", "y_um"]].to_numpy(float)

# (1) spots within 100 µm of the centre spot
ax = axes[0]
spot_map(ax, toy, np.where(toy.index.isin(nb), BLUE, GREY), size=420,
         title="1. Spots within 100 µm of the centre spot (blue)")
ax.scatter(*xy_centre, s=420, facecolors="white", edgecolors="black", linewidths=1.5)
ax.add_patch(plt.Circle(xy_centre, 100, fill=False, ls="--", color=INK))
ax.annotate("centre", xy_centre, ha="center", va="center", fontsize=7)
ax.annotate("exactly 100 µm away: counted (6 spots)\nthe centre is not its own neighbour",
            (0, 290), ha="center", fontsize=8, color=INK)
ax.annotate("isolated spot:\nempty result", (700, 80), ha="center", fontsize=8, color=INK)

# (2) distance to the region
ax = axes[1]
spot_map(ax, toy, dist, size=420, cmap=BLUES, title="2. Distance to the region (µm)")
ax.scatter(toy.x_um[region], toy.y_um[region], s=420, facecolors="none", edgecolors=ORANGE, linewidths=2.5)
for b in toy.index:
    ax.annotate(f"{dist[b]:.0f}", toy.loc[b, ["x_um", "y_um"]].to_numpy(float), ha="center", va="center",
                fontsize=7, color="white" if dist[b] > 400 else "black")
ax.annotate("orange ring = region (distance 0)", (0, 290), ha="center", fontsize=8, color=INK)

# (3) neighbourhood fraction
ax = axes[2]
spot_map(ax, toy, frac.fraction.fillna(0), size=420, cmap=BLUES, vmin=0, vmax=1,
         title="3. Neighbours within 100 µm in the category")
ax.scatter(toy.x_um[category], toy.y_um[category], s=420, facecolors="none", edgecolors=ORANGE, linewidths=2.5)
for b in toy.index:
    f = frac.loc[b]
    label = "0/0\nNaN" if f.n_neighbours == 0 else f"{int(f.n_in_category)}/{int(f.n_neighbours)}"
    ax.annotate(label, toy.loc[b, ["x_um", "y_um"]].to_numpy(float), ha="center", va="center", fontsize=7,
                color="white" if f.fraction > 0.5 else "black")
ax.annotate("orange ring = category; label = in category / all neighbours", (0, 290), ha="center", fontsize=8,
            color=INK)
for ax in axes:                                       # room for the notes under the grid
    ax.set_xlim(-470, 800)
    ax.set_ylim(340, -230)
fig.savefig(FIGS / "synthetic_examples.png", bbox_inches="tight")

**Reading this:**
1. With *r* = 100 µm the centre spot gets exactly its **6 touching spots**: they are exactly 100 µm away, and
   the boundary counts. The centre spot itself is not in its own list. The isolated spot gets an **empty** list.
2. Each number is the distance to the **nearest** orange spot. Inside the region it is 0. Distances grow in steps
   of the grid (100, 173, 200 µm…).
3. Each label is "neighbours in the category / all neighbours". An orange spot does **not** count itself. The
   isolated spot has 0 neighbours, so its fraction is **NaN** (undefined), not 0%.

## 2. Real coordinates are not a perfect grid

Our µm coordinates come from the tissue image (`obsm["spatial_um"]`), so neighbouring spots are *about* 100 µm
apart, not exactly. If we ask for "within 100 µm", some true neighbours at 100.2 µm are missed.

In [ ]:
a24 = ad.read_h5ad(ROOT / "data/processed/GSE298774_A24.h5ad")   # filtered spots (stage 1, notebook 03)
spots = spots_table(a24)                                           # one section: A24

xy = spots[["x_um", "y_um"]].to_numpy()
nn = cKDTree(xy).query(xy, k=2)[0][:, 1]                         # distance to the closest other spot
nobody = pd.Series(False, index=spots.index)
counts = {r: neighbourhood_fraction(spots, nobody, r).n_neighbours for r in (100, 101, 150)}

fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"wspace": 0.3})
axes[0].hist(nn, bins=np.arange(99.5, 100.8, 0.05), color=BLUE)
axes[0].axvline(100, color=INK, ls="--")
axes[0].set_xlabel("distance to the closest other spot (µm)")
axes[0].set_ylabel("spots")
axes[0].set_title("A24: neighbouring spots are about 100 µm apart, not exactly")
table = pd.DataFrame({f"r = {r} µm": c.value_counts().sort_index() for r, c in counts.items()}).fillna(0)
table.index.name = "neighbours"
table.plot.bar(ax=axes[1], color=[GREY, ORANGE, BLUE], width=0.8, rot=0)
axes[1].set_ylabel("spots")
axes[1].set_title("How many neighbours each spot gets")
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(FIGS / "A24_neighbour_distances.png", bbox_inches="tight")
table.astype(int)

**Reading this:** with *r* = 100 µm most A24 spots get only **1 neighbour** instead of 6, and even 101 µm
misses many: the touching spots are 99.8–100.6 µm away because of tiny measurement differences. **Choose radii between the rings of the grid**, not on them: the first ring is at
100 µm and the second at ~173 µm, so **150 µm = "the touching spots"**. Spots on the tissue edge or next to
removed spots (QC holes) still have fewer than 6, and that is real.

## 3. A real example on A24 (provisional regions)

There are no confirmed nevus/melanoma labels, so the regions here are **defined by expression and provisional**:

- **Candidate tumour region:** PRAME-high spots (PRAME > 1.5, log-normalised) where at least half of their
  neighbours within 150 µm are also PRAME-high. This uses operation 3 to drop isolated noisy spots. It is
  *not* a confirmed melanoma.
- **"T-cell response" spots:** the cluster with the highest CXCL9 (a gene that attracts T cells). A cluster is a
  group of similar spots, not a confirmed cell type.

Questions: how far are the T-cell response spots from the candidate tumour, and which spots are surrounded by
them?

In [ ]:
prame = pd.Series(a24[:, "PRAME"].X.toarray().ravel(), index=a24.obs_names)
prame_high = prame > 1.5
support = neighbourhood_fraction(spots, prame_high, 150)
tumour = prame_high & (support.fraction >= 0.5)

cxcl9 = pd.Series(a24[:, "CXCL9"].X.toarray().ravel(), index=a24.obs_names)
cxcl9_by_cluster = cxcl9.groupby(a24.obs.leiden, observed=True).mean()
tcell_cluster = cxcl9_by_cluster.idxmax()
tcell = a24.obs.leiden == tcell_cluster

print(f"PRAME-high spots: {prame_high.sum()}  ->  candidate tumour region: {tumour.sum()} spots")
print(f"T-cell response cluster: {tcell_cluster} ({tcell.sum()} spots, mean CXCL9 {cxcl9_by_cluster.max():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
spot_map(axes[0], spots, np.where(prame_high, BLUE, GREY), size=6, title="PRAME-high spots (blue)")
colors = np.select([tcell, tumour], [ORANGE, BLUE], GREY)
spot_map(axes[1], spots, colors, size=6,
         title="Candidate tumour region (blue) and T-cell response spots (orange)")
fig.savefig(FIGS / "A24_regions.png", bbox_inches="tight")

### Operation 2 on A24: distance to the candidate tumour region

In [ ]:
dist = distance_to_region(spots, tumour)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.25})
sc = spot_map(axes[0], spots, dist, size=6, cmap=BLUES, title="Distance to the candidate tumour region (µm)")
fig.colorbar(sc, ax=axes[0], shrink=0.6, label="µm (0 = inside the region)")

bins = np.arange(0, dist.max() + 100, 100)
axes[1].hist(dist[~tcell], bins=bins, color=GREY, label="all other spots", density=True)
axes[1].hist(dist[tcell], bins=bins, color=ORANGE, label=f"T-cell response spots (n = {tcell.sum()})",
             density=True, alpha=0.85)
axes[1].set_xlabel("distance to the candidate tumour region (µm)")
axes[1].set_ylabel("share of spots (per µm)")
axes[1].set_title("Where the T-cell response spots are, compared with all other spots")
axes[1].legend(frameon=False)
axes[1].spines[["top", "right"]].set_visible(False)
fig.savefig(FIGS / "A24_distance_to_region.png", bbox_inches="tight")

summary = pd.DataFrame({
    "spots": [tcell.sum(), (~tcell).sum()],
    "inside the region (0 µm)": [(dist[tcell] == 0).mean(), (dist[~tcell] == 0).mean()],
    "within 150 µm": [(dist[tcell] <= 150).mean(), (dist[~tcell] <= 150).mean()],
    "median distance (µm)": [dist[tcell].median(), dist[~tcell].median()],
}, index=["T-cell response spots", "all other spots"])
summary.round(2)

### Operation 3 on A24: how surrounded is each spot by T-cell response spots?

In [ ]:
frac = neighbourhood_fraction(spots, tcell, 150)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
sc = spot_map(ax, spots, frac.fraction, size=6, cmap=BLUES, vmin=0, vmax=1,
              title="Share of neighbours (within 150 µm) that are T-cell response spots")
fig.colorbar(sc, ax=ax, shrink=0.6, label="fraction of neighbours")
fig.savefig(FIGS / "A24_neighbourhood_fraction.png", bbox_inches="tight")

print(f"candidate tumour spots with at least one T-cell response neighbour: "
      f"{(frac[tumour].n_in_category > 0).mean():.0%}")
print(f"spots with no neighbours at all (fraction NaN): {frac.fraction.isna().sum()}")

### Operation 1 on A24: the neighbourhood of one spot

In [ ]:
# the candidate-tumour spot with the most T-cell response spots around it
example = frac[tumour].n_in_category.idxmax()
nb = spots_within_radius(spots, example, 300)
xy_example = spots.loc[example, ["x_um", "y_um"]].to_numpy(float)
zoom = spots[np.hypot(spots.x_um - xy_example[0], spots.y_um - xy_example[1]) < 600]

fig, ax = plt.subplots(figsize=(6, 6))
colors = np.select([zoom.index.isin(nb) & tcell[zoom.index].to_numpy(), zoom.index.isin(nb)], [ORANGE, BLUE], GREY)
spot_map(ax, zoom, colors, size=110, title="Spots within 300 µm of one candidate-tumour spot")
ax.scatter(*xy_example, s=110, facecolors="white", edgecolors="black", linewidths=1.5)
ax.add_patch(plt.Circle(xy_example, 300, fill=False, ls="--", color=INK))
fig.savefig(FIGS / "A24_within_radius.png", bbox_inches="tight")
print(f"{example}: {len(nb)} spots within 300 µm (blue + orange), "
      f"of which {tcell[nb].sum()} are T-cell response spots (orange)")

## 4. Summary

- The three operations work in µm, inside one section, with the agreed rules, and pass the tests on drawn
  examples with known answers (boundary, no neighbours, the spot itself, two sections, wrong inputs).
- On real data, **choose radii between grid rings** (150 µm for the touching spots), because image-based
  coordinates are about 100 µm apart, not exactly.
- On A24 the operations give concrete, checkable numbers (tables above). They describe **expression-defined,
  provisional regions**: the candidate tumour region is PRAME-based and the T-cell response spots are a cluster.
  They are not confirmed melanoma or confirmed T cells.
- These operations are the building blocks the LLM will choose from; the benchmark reference answers will be
  computed with them.